# Lab: uma chave de identificação para pinguins

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/decision-trees/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar

**Preveja.** São três espécies de pinguins. Você espera que elas apareçam em proporções parecidas? Que acurácia teria um modelo que sempre responde a espécie mais comum?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

peng = fetch_openml(data_id=42585, as_frame=True).frame      # Palmer penguins
print(peng.shape)
print(peng['species'].value_counts().to_dict())
print(peng.isna().sum().to_dict())
print(peng['sex'].value_counts(dropna=False).to_dict())

## Passo 2 — limpar e codificar

**Preveja.** A árvore aceita `NaN` e não precisa de escalonamento. Das três limpezas de costume — imputar, escalonar, codificar categorias —, qual ainda é necessária aqui?

In [ ]:
num = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
peng = peng.dropna(subset=num)                    # as 2 aves sem medida nenhuma
X = peng[num].copy()
X['is_male'] = peng['sex'].astype(object).map({'MALE': 1.0, 'FEMALE': 0.0})   # '_' e desconhecido -> NaN
X = X.join(pd.get_dummies(peng['island'], prefix='island', dtype=float))      # one-hot
y = peng['species'].astype(str)
print(X.shape)
print("missing:", X.isna().sum()[X.isna().sum() > 0].to_dict())
print(X.head(3).to_string())

## Passo 3 — trancar o conjunto de teste e um baseline

**Preveja.** Com 86 pinguins no teste, quanto vale um único erro em pontos de acurácia? Isso muda o quanto você deve confiar na terceira casa decimal?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def cv_report(model):
    r = cross_validate(model, X_train, y_train, cv=cv, return_train_score=True)
    return (f"train {r['train_score'].mean():.3f} · validation "
            f"{r['test_score'].mean():.3f} ± {r['test_score'].std():.3f}")


print("dummy:", cv_report(DummyClassifier(strategy='most_frequent')))

## Passo 4 — uma árvore de profundidade 2, lida como regras

**Preveja.** Uma única pergunta (profundidade 1) consegue separar três espécies? E, com duas perguntas em sequência, que medida você acha que a raiz vai usar?

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

print("depth 1:", cv_report(DecisionTreeClassifier(max_depth=1, random_state=0)))
print("depth 2:", cv_report(DecisionTreeClassifier(max_depth=2, random_state=0)))

tree2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print(tree2.classes_)
print(export_text(tree2, feature_names=list(X.columns), show_weights=True))

plot_tree(tree2, feature_names=list(X.columns), class_names=list(tree2.classes_),
          filled=True, rounded=True, fontsize=8)
plt.show()

**Simulação: a raiz, escolhida por você.** A seção 3 pontuou à mão todos os cortes de uma tabela de 12 clientes. Faça o mesmo com os pinguins: para cada uma das quatro medidas, teste todos os pontos médios entre valores vizinhos em `X_train`, pontue cada corte pelo Gini ponderado (agora com três classes) e guarde o melhor. A pergunta vencedora é a raiz que a árvore do passo 4 escolheu? Por quanto ela vence?

In [ ]:
# sua simulação aqui

## Passo 5 — crescer até o fim

**Preveja.** Sem limite nenhum, a árvore chega a 100% de acurácia no treino? E na validação: ela vai piorar em relação à de profundidade 2, como no Spambase, ou melhorar?

In [ ]:
full = DecisionTreeClassifier(random_state=0)
print("fully grown:", cv_report(full))
full.fit(X_train, y_train)
print(f"depth {full.get_depth()} · {full.get_n_leaves()} leaves for {len(X_train)} penguins")

**Simulação: uma árvore decora até rótulos aleatórios.** Embaralhe as espécies de `y_train` — cada pinguim fica com a espécie de outro, sorteado — e repita a validação cruzada da árvore crescida até o fim. O que acontece com a acurácia de treino? E com a de validação? Compare também o número de folhas com o da árvore com os rótulos verdadeiros.

In [ ]:
# sua simulação aqui

**Para fechar a parte fundamental.** Em uma frase cada: (1) o que faz uma pergunta ser melhor que outra? (2) Por que uma árvore crescida até o fim chega a 100% de acurácia de treino, mesmo com rótulos aleatórios? (3) Como você leria em voz alta um caminho da raiz até uma folha da chave do passo 4?

## Sua vez (fundamental)

Para casa:

1. **Entropia.** Refaça o passo 4 com `criterion='entropy'`. A árvore de profundidade 2 escolhe as mesmas perguntas? E os mesmos limiares?
2. **O ganho de informação à mão.** Na tabela dos 12 clientes, calcule à mão a entropia dos dois filhos de `bill ≤ 82.5` e o ganho de informação do corte. Confira com o 0,476 da seção 3.
3. **Folhas mínimas em vez de profundidade.** Refaça o passo 5 com `min_samples_leaf=5` e com `min_samples_leaf=10`. Quantas folhas a árvore tem e o que acontece com a acurácia de treino e com a de validação?

## Passo 6 — ajustar profundidade, tamanho de folha e poda juntos

**Preveja.** Numa grade de 96 configurações, você espera um vencedor isolado ou várias configurações empatadas dentro do ruído dos folds? Qual delas caberia num cartão?

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {'max_depth': [1, 2, 3, 4, 5, None],
        'min_samples_leaf': [1, 2, 5, 10],
        'ccp_alpha': [0, 0.01, 0.02, 0.05]}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=cv)
search.fit(X_train, y_train)

res = pd.DataFrame(search.cv_results_['params'])
res['validation'] = search.cv_results_['mean_test_score']
res['std'] = search.cv_results_['std_test_score']
res['leaves'] = [DecisionTreeClassifier(random_state=0, **p).fit(X_train, y_train).get_n_leaves()
                 for p in search.cv_results_['params']]
print(f"{len(res)} configurations · best {search.best_params_} · {search.best_score_:.3f}")

sd = res.loc[search.best_index_, 'std']                   # desvio padrão entre folds do melhor
close = res[res['validation'] >= search.best_score_ - sd]
print(f"within one std ({sd:.3f}) of the best: {len(close)} configurations")
print(close.sort_values(['leaves', 'validation'], ascending=[True, False]).head(6).round(3).to_string())

## Passo 7 — os rivais, nos mesmos folds

**Preveja.** A regressão logística traça fronteiras retas em qualquer direção; a árvore, só degraus paralelos aos eixos. Sabendo que Adelie e Chinstrap diferem sobretudo na razão entre o comprimento e a profundidade do bico, quem você espera que vença?

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

chosen = close.sort_values(['leaves', 'validation'], ascending=[True, False]).iloc[0]
params = {'max_depth': None if pd.isna(chosen['max_depth']) else int(chosen['max_depth']),
          'min_samples_leaf': int(chosen['min_samples_leaf']), 'ccp_alpha': float(chosen['ccp_alpha'])}
key = DecisionTreeClassifier(random_state=0, **params)
print("chosen tree:", params)

prep = lambda: [SimpleImputer(strategy='most_frequent'), StandardScaler()]   # estes modelos precisam dos dois
rivals = {
    'logistic': GridSearchCV(make_pipeline(*prep(), LogisticRegression(max_iter=5000)),
                             {'logisticregression__C': [0.01, 0.1, 1, 10, 100]}, cv=cv),
    'k-NN': GridSearchCV(make_pipeline(*prep(), KNeighborsClassifier()),
                         {'kneighborsclassifier__n_neighbors': [1, 3, 5, 9, 15, 25]}, cv=cv),
}
folds = pd.DataFrame({'tree': cross_validate(key, X_train, y_train, cv=cv)['test_score']})
for name, s in rivals.items():
    s.fit(X_train, y_train)
    print(f"{name}: {s.best_params_}")
    folds[name] = [s.cv_results_[f'split{k}_test_score'][s.best_index_] for k in range(5)]
folds['logistic - tree'] = folds['logistic'] - folds['tree']
folds['k-NN - tree'] = folds['k-NN'] - folds['tree']
print(folds.round(3).to_string())
print(pd.DataFrame({'mean': folds.mean(), 'std': folds.std(ddof=0)}).T.round(3).to_string())

**Simulação: dê a diagonal à árvore.** O passo 7 atribuiu a derrota da chave a uma fronteira diagonal entre Adelie e Chinstrap. Teste a explicação: acrescente a razão `culmen_length_mm / culmen_depth_mm` como coluna e refaça a validação cruzada da mesma receita (profundidade 3, `ccp_alpha=0.01`). A acurácia sobe? A razão vira a raiz? Quantas folhas a chave precisa agora?

In [ ]:
# sua simulação aqui

## Passo 8 — o único teste

**Preveja.** A validação cruzada deu 0,961 ± 0,025 para a chave. No teste, você espera um resultado parecido? Que par de espécies deve concentrar os erros?

In [ ]:
from sklearn.metrics import confusion_matrix

key.fit(X_train, y_train)
pred = key.predict(X_test)
print(f"test accuracy {np.mean(pred == y_test):.3f} · misidentified: {np.sum(pred != y_test)} of {len(y_test)}")
print(pd.DataFrame(confusion_matrix(y_test, pred), index=key.classes_, columns=key.classes_))

## Passo 9 — a chave e o que ela usa

**Preveja.** Das 8 colunas, quais você espera ver na chave? Alguma delas pode estar ali por um motivo que não tem nada a ver com biologia?

In [ ]:
print(export_text(key, feature_names=list(X.columns), show_weights=True))
imp = pd.Series(key.feature_importances_, index=X.columns)
print(imp[imp > 0].sort_values(ascending=False).round(3).to_string())

## Passo 10 — quão estável é a chave?

**Preveja.** Se outra equipe medisse pinguins parecidos e rodasse a mesma receita, a chave começaria pela mesma pergunta? E as previsões, mudariam tanto quanto as perguntas?

In [ ]:
rng = np.random.default_rng(0)
roots, preds = [], []
for b in range(100):
    idx = rng.integers(0, len(X_train), len(X_train))             # reamostra bootstrap
    m = DecisionTreeClassifier(random_state=0, **params).fit(X_train.iloc[idx], y_train.iloc[idx])
    roots.append((X.columns[m.tree_.feature[0]], round(m.tree_.threshold[0], 2)))
    preds.append(m.predict(X_test))
roots = pd.DataFrame(roots, columns=['feature', 'threshold'])
print(roots.groupby('feature')['threshold'].agg(['count', 'min', 'max']))
preds = np.array(preds)
disagree = np.mean([np.mean(preds[i] != preds[j]) for i in range(100) for j in range(i + 1, 100)])
print(f"two resampled keys disagree on {disagree:.1%} of the test penguins")
print(f"test penguins on which all 100 keys agree: {np.mean((preds == preds[0]).all(axis=0)):.1%}")

**Simulação: bagging à mão.** O passo 10 mostrou que chaves treinadas em reamostras diferentes discordam. Use isso a seu favor: dentro de cada fold do `cv`, ajuste 100 árvores crescidas até o fim, cada uma numa reamostra bootstrap do treino do fold, e deixe-as votar (a espécie mais frequente entre as 100 previsões). Compare com uma única árvore crescida até o fim, nos mesmos folds.

In [ ]:
# sua simulação aqui

**Para fechar a aula.** Em uma frase cada: (1) por que a chave perdeu para a regressão logística e o que a recuperou em parte? (2) Por que chaves treinadas em reamostras diferentes começam por perguntas diferentes? (3) Por que o voto de muitas árvores erra menos que uma árvore só?

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Entropia na grade.** Acrescente `'criterion': ['gini', 'entropy']` à grade do passo 6. Quantas das configurações a menos de um desvio padrão da melhor usam entropia? A chave escolhida muda?
2. **Uma árvore de regressão.** No CSV de imóveis da Califórnia (`pd.read_csv('https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula02/housing.csv')`), preveja `median_house_value` a partir das colunas numéricas com um `DecisionTreeRegressor`. Ajuste `max_depth` e `min_samples_leaf` com `KFold(5, shuffle=True, random_state=0)` e compare o RMSE da melhor árvore com o de uma `LinearRegression`. Qual é o maior valor que a árvore pode prever? Por quê?
3. **Importância por permutação.** Calcule `permutation_importance(key, X_test, y_test, n_repeats=30, random_state=0)` (de `sklearn.inspection`) e compare com as importâncias por impureza do passo 9. A ilha Dream continua importando? Algum atributo que a chave não usa ganha importância?